# InsightForge AI — Agent 1: Schema Agent
Automatically detects column types, business domain, target variable, and problem type using Gemini AI.


In [ ]:
%pip install pandas google-generativeai
dbutils.library.restartPython()


In [ ]:
class InsightForgeState(TypedDict):
    """
    Shared state passed through all agents in the pipeline.
    Each agent reads what it needs and writes its output.
    No agent modifies another agent's output fields.

    Fields
    ------
    dataset_path     : path to the input CSV file
    gemini_key       : Gemini API key passed at runtime
    raw_df           : original DataFrame as uploaded
    cleaned_df       : DataFrame after cleaning agent runs
    schema_info      : column metadata detected by schema agent
    cleaning_report  : summary of all cleaning actions taken
    eda_results      : statistical analysis from EDA agent
    charts           : list of Plotly figure dicts from viz agent
    insights         : AI generated business insights text
    pdf_path         : path to the generated PDF report
    pipeline_log     : timestamped log of each agent execution
    errors           : list of error messages from any agent
    """
    dataset_path    : str
    gemini_key      : str
    raw_df          : Any
    cleaned_df      : Any
    schema_info     : dict
    cleaning_report : dict
    eda_results     : dict
    charts          : list
    insights        : str
    pdf_path        : str
    pipeline_log    : list
    errors          : list

print("✅ InsightForgeState defined")
print()
print("  State fields:")
fields = [
    ("dataset_path",     "input — path to CSV"),
    ("gemini_key",       "input — API key"),
    ("raw_df",           "Schema Agent reads this"),
    ("cleaned_df",       "Cleaning Agent writes this"),
    ("schema_info",      "Schema Agent writes this"),
    ("cleaning_report",  "Cleaning Agent writes this"),
    ("eda_results",      "EDA Agent writes this"),
    ("charts",           "Visualization Agent writes this"),
    ("insights",         "Insight Agent writes this"),
    ("pdf_path",         "Report Agent writes this"),
    ("pipeline_log",     "every agent appends to this"),
    ("errors",           "every agent appends on failure"),
]
for field, desc in fields:
    print(f"    {field:20} — {desc}")


In [ ]:
def log_event(state: InsightForgeState, agent: str, message: str) -> list:
    """
    Appends a timestamped log entry to the pipeline log.
    Called by every agent on start and completion.

    Parameters
    ----------
    state   : current pipeline state
    agent   : name of the calling agent
    message : what happened

    Returns
    -------
    list : updated pipeline log
    """
    timestamp = datetime.now().strftime("%H:%M:%S")
    entry     = f"[{timestamp}] {agent}: {message}"
    print(f"   {entry}")
    return state["pipeline_log"] + [entry]


def get_gemini_model() -> genai.GenerativeModel:
    """
    Returns a configured Gemini model instance.
    Re-reads the API key from widget each time to handle
    session restarts without needing to re-run setup cells.
    """
    key = dbutils.widgets.get("gemini_key")
    genai.configure(api_key=key)
    return genai.GenerativeModel(GEMINI_MODEL)


def safe_call_gemini(prompt: str, agent_name: str) -> str:
    """
    Wraps a Gemini API call with error handling.
    Cleans the response text to remove characters that
    fpdf2 cannot render with standard Helvetica font.

    Parameters
    ----------
    prompt     : the full prompt string to send
    agent_name : name of the calling agent for logging

    Returns
    -------
    str : cleaned response text or error message
    """
    try:
        m        = get_gemini_model()
        response = m.generate_content(prompt)
        text     = response.text

        # Remove characters unsupported by Helvetica in fpdf2
        replacements = {
            "\u2014": "-",    # em dash
            "\u2013": "-",    # en dash
            "\u2012": "-",    # figure dash
            "\u2011": "-",    # non-breaking hyphen
            "\u2010": "-",    # hyphen
            "\u2022": "-",    # bullet
            "\u2023": "-",    # triangle bullet
            "\u2043": "-",    # hyphen bullet
            "\u2018": "'",    # left single quote
            "\u2019": "'",    # right single quote
            "\u201a": "'",    # single low quote
            "\u201c": '"',    # left double quote
            "\u201d": '"',    # right double quote
            "\u201e": '"',    # double low quote
            "\u2026": "...",  # ellipsis
            "\u00a0": " ",    # non-breaking space
            "\u00b7": "-",    # middle dot
            "\u2015": "-",    # horizontal bar
        }
        for char, replacement in replacements.items():
            text = text.replace(char, replacement)

        # Final safety pass — replace remaining non-latin-1 chars
        text = text.encode("latin-1", errors="replace").decode("latin-1")
        return text

    except Exception as e:
        logger.warning(
            f"Gemini call failed in {agent_name}: {str(e)[:100]}"
        )
        print(f"   ⚠️  Gemini call failed in {agent_name}: {e}")
        return f"[Gemini error in {agent_name}: {str(e)}]"


print("✅ Utility functions defined")
print("   log_event()        — timestamped pipeline logging")
print("   get_gemini_model() — safe model initialisation")
print("   safe_call_gemini() — error handled API call with font cleaning")


In [ ]:
def schema_agent(state: InsightForgeState) -> dict:
    """
    Agent 1 — Schema Agent
    ----------------------
    Responsibility: understand what the dataset is about.

    Reads  : state["raw_df"]
    Writes : state["schema_info"]
             state["pipeline_log"]

    Uses Gemini to detect the data domain, target variable,
    and industry from column names and sample values.
    Does NOT modify the DataFrame.
    """
    agent_name = "Schema Agent"
    print(f"\n{'─' * 55}")
    print(f"🔵 {agent_name} starting...")

    df     = state["raw_df"]
    errors = state["errors"]
    log    = log_event(state, agent_name, "started")

    logger.info(
        f"{agent_name} started — "
        f"{df.shape[0]} rows x {df.shape[1]} cols"
    )

    try:
        # ── Structural analysis ───────────────────────────────
        numeric_cols     = list(df.select_dtypes(include="number").columns)
        categorical_cols = list(df.select_dtypes(include="object").columns)
        datetime_cols    = list(df.select_dtypes(include="datetime").columns)
        missing_by_col   = df.isnull().sum().to_dict()
        missing_pct      = (
            df.isnull().sum() / len(df) * 100
        ).round(2).to_dict()

        # ── Gemini domain detection ───────────────────────────
        prompt = f"""
You are a data scientist. Analyse these dataset details.

Column names : {list(df.columns)}
Data types   : {df.dtypes.astype(str).to_dict()}
Sample row   : {df.iloc[0].to_dict()}
Missing cols : {[c for c, v in missing_by_col.items() if v > 0]}

Reply in EXACTLY this format, one value per line:
DOMAIN: [type of data e.g. passenger records, sales transactions]
TARGET: [most likely target or outcome column name only]
INDUSTRY: [industry e.g. Transportation, Retail, Healthcare]
SUMMARY: [one sentence describing this dataset]
"""
        response = safe_call_gemini(prompt, agent_name)

        # ── Parse Gemini response ─────────────────────────────
        domain   = "Unknown"
        target   = ""
        industry = "Unknown"
        summary  = "No summary available"

        for line in response.split("\n"):
            line = line.strip()
            if line.startswith("DOMAIN:")  : domain   = line.replace("DOMAIN:",   "").strip()
            if line.startswith("TARGET:")  : target   = line.replace("TARGET:",   "").strip()
            if line.startswith("INDUSTRY:"): industry = line.replace("INDUSTRY:", "").strip()
            if line.startswith("SUMMARY:") : summary  = line.replace("SUMMARY:",  "").strip()

        # ── Build schema info dict ────────────────────────────
        schema_info = {
            "columns"         : list(df.columns),
            "dtypes"          : df.dtypes.astype(str).to_dict(),
            "numeric_cols"    : numeric_cols,
            "categorical_cols": categorical_cols,
            "datetime_cols"   : datetime_cols,
            "row_count"       : int(df.shape[0]),
            "col_count"       : int(df.shape[1]),
            "missing_by_col"  : missing_by_col,
            "missing_pct"     : missing_pct,
            "domain"          : domain,
            "target_variable" : target,
            "industry"        : industry,
            "summary"         : summary,
        }

        log = log_event(
            state, agent_name,
            f"done — domain={domain}, target={target}"
        )
        logger.info(
            f"{agent_name} complete — "
            f"domain={domain}, target={target}"
        )
        print(f"   Domain   : {domain}")
        print(f"   Target   : {target}")
        print(f"   Industry : {industry}")
        print(f"   Summary  : {summary[:60]}...")
        print(f"✅ {agent_name} complete")

        return {
            "schema_info"  : schema_info,
            "pipeline_log" : log,
            "errors"       : errors
        }

    except Exception as e:
        logger.error(f"{agent_name} FAILED — {str(e)}")
        msg = f"{agent_name} failed: {str(e)}"
        print(f"   ❌ {msg}")
        return {
            "schema_info"  : {},
            "pipeline_log" : log_event(state, agent_name, f"FAILED — {e}"),
            "errors"       : errors + [msg]
        }

print("✅ schema_agent() defined")
